# Chapter 3.2 · Agent Memory

The chapter's two sessions, run for real: on Monday Dana Whitfield at Meridian Pension Trust asks for the Deere and Caterpillar comparison and says she prefers tables; a week later she writes "And NVIDIA?" in a new session. The same task twice, once with the **OpenAI Agents SDK** and once with **LangGraph**, both on `glm-5.3-flash` via Lumen.

**Before you run:** the same three steps as chapter 1. Sign in at [lumen.ncsa.illinois.edu](https://lumen.ncsa.illinois.edu/chat), create an API key on your [profile page](https://lumen.ncsa.illinois.edu/profile), and add it to Colab as a secret named `LUMEN_API_KEY` with notebook access enabled.

In [ ]:
%pip install -q openai-agents langchain langgraph langchain-openai

In [ ]:
import json, os
from google.colab import userdata

LUMEN_API_KEY = userdata.get("LUMEN_API_KEY")
LUMEN_URL = "https://lumen.ncsa.illinois.edu/v1"
MODEL = "glm-5.3-flash"  # highly preferred for this workshop

# The firm's figures, the same as the textbook's tools.py. Tesla is missing on purpose.
FINANCIALS = {
    "DE":   "Deere grew revenue 6.4% YoY to $13.8B in Q2-2026",
    "CAT":  "Caterpillar grew revenue 3.1% YoY to $16.9B in Q2-2026",
    "NVDA": "NVIDIA grew revenue 58.0% YoY to $52.4B in Q2-2026",
    "AAPL": "Apple grew revenue 5.0% YoY to $96.1B in Q2-2026",
    "MSFT": "Microsoft grew revenue 17.0% YoY to $76.3B in Q2-2026",
}

## Part A · The OpenAI Agents SDK

- **Short-term memory** is a *session*. `SQLiteSession("id", "memory.db")` stores every turn under an id, and each run with that session replays them.
- **Long-term memory** is not built in. Here it is a `remember` tool that writes a JSON file, and an instructions function that reads the file back into the system prompt on every run.

Colab already runs an event loop, so the cells use `await Runner.run(...)`.

In [ ]:
from agents import Agent, Runner, SQLiteSession, function_tool, OpenAIChatCompletionsModel, set_tracing_disabled
from openai import AsyncOpenAI

set_tracing_disabled(True)   # tracing would send run logs to OpenAI; we are on Lumen
model = OpenAIChatCompletionsModel(model=MODEL, openai_client=AsyncOpenAI(base_url=LUMEN_URL, api_key=LUMEN_API_KEY))
NOTES = "meridian_notes.json"

@function_tool
def get_financials(ticker: str) -> str:
    """Quarterly revenue and YoY growth for one ticker (DE, CAT, NVDA, AAPL, MSFT)."""
    return FINANCIALS.get(ticker.upper(), f"no data for {ticker}")

@function_tool
def remember(note: str) -> str:
    """Save one short note about Meridian Pension Trust for future sessions: a preference, or a figure with its source."""
    notes = json.load(open(NOTES)) if os.path.exists(NOTES) else []
    json.dump(notes + [note], open(NOTES, "w"))
    return "saved"

def instructions(ctx, agent):
    notes = json.load(open(NOTES)) if os.path.exists(NOTES) else []
    return ("You draft replies for Meridian Pension Trust, a client of Champaign Capital Research. "
            "Look figures up with get_financials and cite them. When the client states a preference, save it with remember.\n"
            "What we remember:\n" + ("\n".join(f"- {n}" for n in notes) or "(nothing yet)"))

analyst = Agent(name="analyst", instructions=instructions, model=model, tools=[get_financials, remember])
if os.path.exists(NOTES):
    os.remove(NOTES)   # start the week with an empty notebook

In [ ]:
# Monday: one session holds the whole conversation
monday = SQLiteSession("meridian-monday", "memory.db")
r = await Runner.run(analyst, "Compare Deere's revenue growth with Caterpillar's last quarter.", session=monday)
print("Monday 1:", r.final_output)
r = await Runner.run(analyst, "I prefer tables, and keep it under 150 words. Please remember that.", session=monday)
print("\nMonday 2:", r.final_output)

print("\nTurns stored in Monday's session:", len(await monday.get_items()))
print("The agent's notebook:", json.load(open(NOTES)) if os.path.exists(NOTES) else "empty")

In [ ]:
# Next Monday: a new session, so only the notebook carries over
next_week = SQLiteSession("meridian-next-monday", "memory.db")
r = await Runner.run(analyst, "And NVIDIA?", session=next_week)
print(r.final_output)

Read the answer. Did the table preference carry over? Did the Deere and Caterpillar figures? Only what the agent chose to write in its notebook survived the new session.

## Part B · LangGraph

- **Short-term memory** is a *checkpointer*. `InMemorySaver()` saves each conversation's state under a `thread_id`; call again with the same id and it continues.
- **Long-term memory** is a *store*. `InMemoryStore()` holds notes in namespaces such as `("meridian", "notes")`, shared by every thread. A tool reaches it through `ToolRuntime`.

`create_agent` is LangChain's ready-made agent, built on LangGraph.

In [ ]:
import uuid
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore

llm = ChatOpenAI(model=MODEL, base_url=LUMEN_URL, api_key=LUMEN_API_KEY)

@tool("get_financials")
def lc_get_financials(ticker: str) -> str:
    """Quarterly revenue and YoY growth for one ticker (DE, CAT, NVDA, AAPL, MSFT)."""
    return FINANCIALS.get(ticker.upper(), f"no data for {ticker}")

@tool("remember")
def lc_remember(note: str, runtime: ToolRuntime) -> str:
    """Save one short note about Meridian Pension Trust for future sessions: a preference, or a figure with its source."""
    runtime.store.put(("meridian", "notes"), str(uuid.uuid4()), {"note": note})
    return "saved"

checkpointer = InMemorySaver()   # short-term: each thread's messages
store = InMemoryStore()          # long-term: notes shared by every thread
lg_analyst = create_agent(llm, tools=[lc_get_financials, lc_remember], checkpointer=checkpointer, store=store,
                          system_prompt="You draft replies for Meridian Pension Trust, a client of Champaign Capital Research. "
                                        "Look figures up with get_financials and cite them. When the client states a preference, save it with remember.")

In [ ]:
# Monday: one thread holds the whole conversation
monday = {"configurable": {"thread_id": "meridian-monday"}}
r = lg_analyst.invoke({"messages": [("user", "Compare Deere's revenue growth with Caterpillar's last quarter.")]}, monday)
print("Monday 1:", r["messages"][-1].content)
r = lg_analyst.invoke({"messages": [("user", "I prefer tables, and keep it under 150 words. Please remember that.")]}, monday)
print("\nMonday 2:", r["messages"][-1].content)
print("\nMessages in Monday's thread:", len(r["messages"]))

notes = [item.value["note"] for item in store.search(("meridian", "notes"))]
print("Notes in the store:", notes)

In [ ]:
# Next Monday: a new thread, so only the notes in the store carry over
next_week = {"configurable": {"thread_id": "meridian-next-monday"}}
notes = [item.value["note"] for item in store.search(("meridian", "notes"))]
r = lg_analyst.invoke({"messages": [("system", "What we remember:\n" + "\n".join(f"- {n}" for n in notes)),
                                    ("user", "And NVIDIA?")]}, next_week)
print(r["messages"][-1].content)

## Exercise

1. In Part A, run Monday's session, then next Monday's with a *new* `SQLiteSession` id but **without** the `remember` tool (take it out of `tools=`). What does the agent remember, and why? Put the tool back and try again.
2. In Part B, look inside the store after Monday (`store.search(("meridian", "notes"))`). Change `lc_remember` so every note must include a source, and refuse to save one without it (return a message saying why).
3. In three sentences: one thing you would let an agent remember about a client, one you would only let the harness save, and one you would never keep at all, with a reason for each.